# Problem Statement
Roman numerals are represented by seven different symbols: I, V, X, L, C, D and M.

Symbol       Value
I             1
V             5
X             10
L             50
C             100
D             500
M             1000
For example, 2 is written as II in Roman numeral, just two ones added together. 12 is written as XII, which is simply X + II. The number 27 is written as XXVII, which is XX + V + II.

Roman numerals are usually written largest to smallest from left to right. However, the numeral for four is not IIII. Instead, the number four is written as IV. Because the one is before the five we subtract it making four. The same principle applies to the number nine, which is written as IX. There are six instances where subtraction is used:

I can be placed before V (5) and X (10) to make 4 and 9. 
X can be placed before L (50) and C (100) to make 40 and 90. 
C can be placed before D (500) and M (1000) to make 400 and 900.
Given a roman numeral, convert it to an integer.

 

Example 1:

Input: s = "III"
Output: 3
Explanation: III = 3.

Example 2:

Input: s = "LVIII"
Output: 58
Explanation: L = 50, V= 5, III = 3.

Example 3:

Input: s = "MCMXCIV"
Output: 1994
Explanation: M = 1000, CM = 900, XC = 90 and IV = 4.
 

Constraints:

1 <= s.length <= 15
s contains only the characters ('I', 'V', 'X', 'L', 'C', 'D', 'M').
It is guaranteed that s is a valid roman numeral in the range [1, 3999].

# My reasoning
It seems that the pattern is that one number is able to subtract from the two following larger ones. The last number that was able to be subtracted is the one that subtracts the following 2. 
1. `I` can subtract to both `V` and `X`
2. `X` can subtract to both `L` and `C`
3. `C` can subtract to both `D` and `M`

While in general the leftmost value is the larger one, when it comes to subtraction there is a smaller number before. As there are only 6 cases I might be able to store the combinations of subtraction. I imagine a sliding window that has a size of minimum 1 and maximum 2 and that uses the values in the window as a key to unlock a value in a dictionary. 

In [1]:
roman_to_integer_dic = {"I": 1, "V": 5, "X": 10, "L": 50, "C": 100, "D": 500, "M": 1000}
print(roman_to_integer_dic.items())

dict_items([('I', 1), ('V', 5), ('X', 10), ('L', 50), ('C', 100), ('D', 500), ('M', 1000)])


A problem comes with I, as I can have up to 3 of them... could I have 3 of X? For example, if I wanted 80 the roman string would be LXXX? It seems right, but I think that should occur only with values that start with 1, that is I, X, C and M. I can go only to 3999, which makes me believe that the rule of 3 is applying here, and that in order to represent the 4000 I would need an extra letter that represented 5000.

I know that the string lenght cannot be greater than 15. The window can actually be of size 3 but that only occurs when the same number is being repeated. I will start with the logic of numbers

In [2]:
sliding_window = [0] * 3 # As there seems to not be a zero in Roman I am certain that any zero is an impossible value
print(sliding_window)

[0, 0, 0]


Lets try with the first input

Input: s = "III"
Output: 3
Explanation: III = 3.

In [3]:
s0 = "III"
# As s must have at least one number I will take advantage of that

def romanToInteger(roman_str):
    frst_char = roman_str[0]
    sliding_window[0] = frst_char
    total_value = roman_to_integer_dic[frst_char]
    for i in range(1, len(roman_str)):
        if frst_char == roman_str[i]:
            total_value += roman_to_integer_dic[frst_char]
            sliding_window[i] = roman_to_integer_dic[frst_char]
    return total_value

print(romanToInteger(s0))

3


This function works at least when it comes to three identical letters, but I also need to think about the other two cases:
1. when the following letter is bigger that means we're talking about a substraction. 
2. When the following letter is smaller, then we're talking about an addition

In [10]:
def romanToInteger(roman_str):
    frst_char = roman_str[0]
    sliding_window[0] = frst_char
    sliding_counter = 0
    total_value = roman_to_integer_dic[frst_char]
    for i in range(1, len(roman_str)):
        print(f"total value: {total_value}  ;  iteration {i}")
        if frst_char == roman_str[i]:
            total_value += roman_to_integer_dic[frst_char]
            sliding_window[i] = roman_to_integer_dic[frst_char]
            sliding_counter += 1
        elif roman_to_integer_dic[frst_char] < roman_to_integer_dic[roman_str[i]]:
            total_value += roman_to_integer_dic[roman_str[i]] - roman_to_integer_dic[frst_char]
            if i + 2 < len(roman_str) - 1:
                frst_char = roman_str[i + 2] 
        else:
            total_value += roman_to_integer_dic[roman_str[i]] + roman_to_integer_dic[frst_char]
            if i + 2 < len(roman_str) - 1:
                frst_char = roman_str[i + 2] 
    return total_value


Trying the second case we have that 

Input: s = "LVIII"
Output: 58
Explanation: L = 50, V= 5, III = 3.

In [11]:
s1 = "LVIII"
print(romanToInteger(s1))

total value: 50  ;  iteration 1
total value: 105  ;  iteration 2
total value: 106  ;  iteration 3


IndexError: list assignment index out of range

My logic seems to be off, as I am getting 

```bash
total value: 50  ;  iteration 1
total value: 105  ;  iteration 2
total value: 106  ;  iteration 3
```

when I print the total_value for each iteration. 

I see the flaw in my logic now: I treated each letter as if influence one another in all cases, but that is only real when they're "continuos" to one another... actually my problem has more to do with an incorrect use of sumation. While I want to sum L + V, the reality is that it is the whole modification of VIII the one that I want to sum. I do think I should stick with with my sliding window approach... In this case the window can become of lenght 4. 

In [ ]:
sliding_window = [0] * 4

def romanToInteger(roman_str):
    sliding_window[0] = roman_to_integer_dic[roman_str[0]]
    sliding_counter = 1
    total = 0
    for i in range(1, len(roman_str)):
        if roman_to_integer_dic[sliding_window[i-1]] == roman_to_integer_dic[roman_str[i]]:
            sliding_window[i] = roman_to_integer_dic[roman_str[i]]
            sliding_counter += 1
        elif roman_to_integer_dic[sliding_window[i-1]] < roman_to_integer_dic[roman_str[i]]:
            sliding_window[i] = roman_to_integer_dic[i - 1] - roman_to_integer_dic[i]
            sliding_counter += 1
    return sum(sliding_window)

I know this previous function is poorly written, but I have noticed that I might directly translate the symbols to numbers and sum them by pairs depending on two cases: 
1. The left number is equal or greater to the right number then it is an addition
2. The left number is lower to the right number then it is an subtraction

For example, `LVIII` is literally `[50, 5, 1, 1, 1]`. I start with the value of the left most number. In this case that is 50. 
0. 50 is greater than 5, then I add 5 (total is 55)
1. 5 is greater than 1, then I add 1 (total is 56)
2. 1 is equal to 1, then I add 1 (total is 57)
3. 1 is equal to 1, then I add 1 (total is 58)

I arrive to the correct result, just like the first case

In [14]:
def romanToInteger(roman_str):
    roman_values = [0] * len(roman_str)
    index = 0
    for letter in roman_str:
        roman_values[index] = roman_to_integer_dic[letter]
        index += 1
    total = roman_values[0]
    for i in range(1, len(roman_values)):
        if roman_values[i - 1] >= roman_values[i]:
            total += roman_values[i]
    return total

Lets check if it works as I expect with the first 2 cases

In [16]:
print(romanToInteger(s0))
print(romanToInteger(s1))
print(roman_to_integer_dic)

3
58
{'I': 1, 'V': 5, 'X': 10, 'L': 50, 'C': 100, 'D': 500, 'M': 1000}


It works! Now lets see if I can get to the third case

Example 3:

Input: s = "MCMXCIV"
Output: 1994
Explanation: M = 1000, CM = 900, XC = 90 and IV = 4. 

Doing the calculations in paper it seems that this logic is not enough when there are subtractions... but I could create the sliding window for subtraction: if I see a lower value, I continue one step further to see if that value subtracts to another. If that is true, then I merge those two values. I could in fact do a first pass to merge those values and other one to sum them. I can worry about efficiency later. This works!

In [17]:
def romanToInteger(roman_str):
    if len(roman_str) == 1:
        return roman_to_integer_dic[roman_str]
    else: 
        roman_values = [0] * len(roman_str)
        index = 0
        for letter in roman_str:
            roman_values[index] = roman_to_integer_dic[letter]
            index += 1
        for i in range(1, len(roman_values)):
            if roman_values[i - 1] < roman_values[i]:
                roman_values[i - 1] = roman_values[i] - roman_values[i - 1]
                roman_values[i] = 0
        return sum(roman_values)

As it is a sum, and the sum is commutative, I can simply add all the values in `roman_values` to get to the correct answer, that is I just need to check when there is a substraction. Lets check with all 3 cases 

In [18]:
s2 = "MCMXCIV"
print(romanToInteger(s0))
print(romanToInteger(s1))
print(romanToInteger(s2))

3
58
2016


Now I see the problem! As I put a 0, that 0 is constantly comparing with the following number, and as it will always be lower than that following number the subtractions aren't being made when it is needed to. What I need is when there is a subtraction I should put probably shift the pointer one more step to the right (while also changing one of the values that I subtracted to 0). 